# Module 24 — Tkinter

The same numbers as modules 21, 22 and 23, in a window on your own machine. No
browser, no port, no HTTP.

```console
uv run python 24_tkinter/app.py
```

A window opens. Nothing to open in a browser, no address to type — and **Ctrl+C does
not reliably stop it.** Closing the window does. That is not a quirk; it is the
module:

> **The program does not run. It waits.**

`app.py` ends with `root.mainloop()`, and nothing written after that line executes
while the window is on screen. Control is handed to Tk once, at startup, and comes
back when the window is destroyed. Everything in between happens because Tk called
it.

**This notebook needs a graphical session** — the only one in the course that does.
The cell below says whether you have one. If it prints `False`, the window cells
raise a `TclError`; on Linux, `xvfb-run -a uv run jupyter lab` gives you a display
that is not attached to a screen. The exercises' tests skip themselves in that case
instead of failing; the notebook does not, because a prediction you cannot run is
worth nothing.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

from display import has_display, use_bundled_tcl  # noqa: E402

# Has to run before the first Tk(). display.py says why, at length: tkinter wraps a C
# library that looks for its own script files under sys.prefix, and inside a virtual
# environment that is the virtual environment -- where uv did not put them.
use_bundled_tcl()

print("display available:", has_display())

import tkinter  # noqa: E402

print("Tk version:", tkinter.TkVersion)

`Tk 8.6` on macOS and `Tk 9.0` on Linux, for the same Python. It is a different C
library on each platform, which is the first hint that this module is not about
Python at all.

## The loop has to turn

`root.after(0, f)` reads like "call `f` now". It is not.

In [ ]:
root = tkinter.Tk()
root.withdraw()  # off the screen; the event loop works exactly the same

log: list[str] = []
root.after(0, lambda: log.append("one"))
root.after(0, lambda: log.append("two"))

In [ ]:
# Two callbacks are scheduled, both due immediately. What is in the log?
assert log == ...

In [ ]:
# update() is the event loop, run once by hand: it takes what is due off the queue,
# runs it, and returns. Unlike mainloop(), it does not block.
root.update()
print(log)

`after` does not mean "in n milliseconds". It means **not before** n milliseconds, and
then whenever the loop is free. It returned an identifier and put the function on a
queue; the thing that takes work off that queue is the event loop, and while your
script is running, the loop is not.

There is one thread. It was executing this notebook.

### `mainloop()` is the same loop, with no way out

The only thing that ends `mainloop()` is the window being destroyed. So the
destruction has to be scheduled **before** control is handed over — after
`mainloop()`, you cannot schedule anything.

In [ ]:
import time

log.clear()
root.after(50, lambda: log.append("during mainloop"))
root.after(100, root.destroy)

start = time.perf_counter()
root.mainloop()
print(f"mainloop returned after {time.perf_counter() - start:.2f} s, log = {log}")

## A slow callback freezes the window

A callback runs *in* the loop, not alongside it. While it works, the loop does not:
nothing is redrawn, no click is noticed, and no other callback runs.

`slow` sleeps for 300 ms. `quick` is due at 10 ms and does nothing at all.

In [ ]:
root = tkinter.Tk()
root.withdraw()

order: list[tuple[str, float]] = []
start = time.perf_counter()


def slow() -> None:
    order.append(("slow start", time.perf_counter() - start))
    time.sleep(0.3)
    order.append(("slow end", time.perf_counter() - start))


def quick() -> None:
    order.append(("quick", time.perf_counter() - start))


root.after(0, slow)
root.after(10, quick)
root.after(600, root.destroy)
# A second, unconditional destroy. If anything above went wrong, mainloop() would
# otherwise block this notebook until somebody killed the kernel -- and a check that
# hangs is worse than a check that fails.
root.after(3000, root.destroy)
root.mainloop()

for name, when in order:
    print(f"  {name:12} {when * 1000:6.0f} ms")

`quick` was due at 10 ms and did not get its turn until `slow` had returned. One
thread, and `slow` was holding it. Your own numbers will differ -- which is why
nothing in this module asserts one.

Three things a user notices while that is happening:

1. **The window does not repaint.** Not "shows stale data" — it does not redraw.
   Drag another window across it and the uncovered part stays blank, because a
   redraw is an event and events are not being processed.
2. **Clicks are held, not refused.** The windowing system queues unread input; a
   program that is not reading its queue is not told about anything, and is not told
   that it was not told. The presses arrive when the loop comes back.
3. **The operating system marks the program as not responding** and offers to kill
   it.

So a thirty-second HTTP fetch must not go in a callback. The shape that works is a
callback that starts the work, returns immediately, and uses `after` to get back:

```python
def start(self):
    self.button.config(state="disabled")          # cannot be pressed twice
    threading.Thread(target=self.fetch, daemon=True).start()
    self.root.after(50, self.check)

def check(self):
    if self.results.empty():
        self.root.after(50, self.check)           # not done: ask again in 50 ms
        return
    self.show(self.results.get())
    self.button.config(state="normal")
```

Each turn costs a fraction of a millisecond, so the loop stays free. Note that
`check` **reschedules itself** — that is the shape of every periodic job in a GUI,
and it is why `after` exists.

## What a callback looks like, and why

`refresh` in `app.py` takes no arguments and returns nothing:

```python
def refresh(self) -> None:
    ...
```

Both halves follow from one fact: **you do not call this function, the event loop
does.**

- **No arguments**, because the loop has nothing to pass. It knows an event happened
  and which callback was registered; it does not know what your function would want.
- **No return value**, because the loop has nowhere to put one. The call came off a
  queue and there is no caller waiting. A callback that computed something and
  returned it would be throwing it away.

So everything a callback needs it reads from `self`, and everything it produces goes
into a widget. Which is why `app.py` is a **class**: `refresh` runs long after
`__init__` returned and still has to find the listbox. A local variable is gone by
then.

## The parentheses that ran the callback

`command=` wants a function. This is module 14's distinction between a function and
its return value, and here getting it wrong is silent.

In [ ]:
root = tkinter.Tk()
root.withdraw()

pressed: list[str] = []


def note() -> None:
    pressed.append("press")


wrong = tkinter.Button(root, text="wrong", command=note())  # called it, here, now
right = tkinter.Button(root, text="right", command=note)  # handed it over
wrong.pack()
right.pack()
root.update()

print("after building two buttons:", pressed)

In [ ]:
# `note()` already ran once during construction. Now press the wrong button twice.
wrong.invoke()
wrong.invoke()
assert pressed == ...

In [ ]:
print("wrong button, twice:", pressed)
right.invoke()
print("right button, once :", pressed)
print("what the wrong button was given:", repr(wrong.cget("command")))

`note()` returned `None`, and **Tk accepts `None` as "no command"**. No error, no
warning: a button that looks identical to the working one and does nothing. The empty
string in the last line is Tk's way of saying there is nothing registered.

This is the most common bug in tkinter code, and exercise 03 is it.

## A StringVar is not a str

`tkinter.StringVar` is a handle on a value that Tcl also holds. Bind a widget to one
and they stay in step **in both directions**, with nobody calling a redraw. It is
module 22's `session_state` one layer further down.

`trace_add("write", f)` runs `f` on every write. Predict what it sees.

In [ ]:
root = tkinter.Tk()
root.withdraw()

# `master=root` is not decoration here. A Var built without one attaches to Tk's
# *default* root -- the first Tk() created in the process -- and cells above have
# created and destroyed several. Without it this cell silently reads and writes a
# variable belonging to an interpreter that no longer exists. A program with one
# window never meets this; a notebook and a test suite do.
var = tkinter.StringVar(master=root, value="85.0")
seen: list[str] = []
# Tk passes the callback three arguments -- an internal name, an index, the operation.
# None of them is useful here, and `*_` says so.
var.trace_add("write", lambda *_: seen.append(var.get()))

entry = tkinter.Entry(root, textvariable=var)
entry.pack()
root.update()
print("entry shows:", entry.get())

var.set("90.0")
root.update()
print("after var.set('90.0'), entry shows:", entry.get())

entry.delete(0, "end")
entry.insert(0, "42")
root.update()
print("after editing the entry, var says:", var.get())

In [ ]:
# Two changes were made above. What did the trace callback see, in order?
assert seen == ...

In [ ]:
print(seen)

Three entries for two changes, and the middle one is **empty**.

`entry.delete(0, "end")` followed by `entry.insert(0, "42")` is two writes, and after
the first one the variable holds `""`. Any listener that converts on every write meets
`float("")` and raises. That is why validation belongs where the value is *used*, not
on every keystroke.

## An Entry holds text, and that is all it holds

Module 23 had the framework refuse a bad value before the function ran, and name the
field. There is nothing here that does that. An `Entry` holds a string; whoever reads
it converts it, and whoever converts it handles the failure.

In [ ]:
from logic import verdict  # noqa: E402

from sensorreport import load_readings  # noqa: E402

hall = [r for r in load_readings() if r.location == "Hall"]

for typed in ["85.0", "90", "85,0", "warm", ""]:
    try:
        limit = float(typed)
    except ValueError:
        print(f"  {typed!r:8} -> {typed!r} is not a number")
        continue
    print(f"  {typed!r:8} -> {verdict(hall, limit)}")

`'85,0'` is refused here for the same reason module 23 refused it — `float` has one
reading of a string or none. What is different is **who** refused it and **when**: in
module 23 it was the framework, before the function; here it is a `try` block you
wrote, inside the callback, and if you forget it the user gets a traceback on a stream
they cannot see.

Which is the next thing.

## The exception nobody sees

A script that raises stops, and the shell gets a non-zero exit code. A callback that
raises stops nothing.

In [ ]:
root = tkinter.Tk()
root.withdraw()

done: list[str] = []


def broken() -> None:
    raise ValueError("the callback is broken")


def works() -> None:
    done.append("pressed")


bad = tkinter.Button(root, text="bad", command=broken)
good = tkinter.Button(root, text="good", command=works)
bad.pack()
good.pack()
root.update()

# No try around this. If the exception came through, the cell would end here.
bad.invoke()
print("after the bad button:", done)

good.invoke()
print("after the good button:", done)

raised = False
try:
    bad.invoke()
except Exception:
    raised = True
print("invoke() re-raised:", raised)

The traceback is above this cell's output, or in the terminal that started Jupyter —
it went to **stderr**, and nothing in the program's own output mentions it.

And the exit code:

```console
$ uv run python 24_tkinter/solutions/solution_06.py > /dev/null
$ echo $?
0
```

**Zero.** A GUI whose every button raises is a program that reports success. Three
things read that number and would be fooled: a shell `&&` chain, a service manager
with `Restart=on-failure`, and this repository's own CI, where a step that exits 0 is
a green step.

Tk catches it **on purpose**, and the design is defensible: the alternative is that
one broken button tears down the loop, the window vanishes, and a user loses whatever
they had typed. Flask does not exit when one request raises either. The difference is
where the report goes — Flask's 500 reaches the person who caused it and lands in an
access log; Tk's traceback goes to a stream that, for a program started by
double-clicking an icon, is attached to nothing.

The fix is yours to write, and it is module 14 plus module 09:

```python
def guarded(method):
    # It decorates a *method*, so `self` is a parameter of the wrapper. Without it
    # the except branch would replace the ValueError with a NameError -- the same
    # failure, one layer further in.
    @functools.wraps(method)
    def wrapper(self, *args, **kwargs):
        try:
            return method(self, *args, **kwargs)
        except Exception:
            logging.exception("%s failed", method.__name__)
            self.status.set("something went wrong -- see the log")
    return wrapper
```

## Why there is no test client

Modules 21, 22 and 23 each had one: `test_client()`, `AppTest`, `TestClient`. All
three work because the application's input is **data a test can construct** — an HTTP
request is bytes, and `AppTest` sets a widget's value and re-runs the script.

Tkinter's input is not data. It is events from the **window manager**. Here is the
same simulated keypress three times, differing only in whether the window is on
screen:

In [ ]:
def try_a_keypress(arrange) -> list[str]:
    """Bind <Return> on an Entry, fire it with event_generate, report what happened."""
    root = tkinter.Tk()
    arrange(root)
    seen: list[str] = []
    entry = tkinter.Entry(root)
    entry.pack()
    entry.insert(0, "90.0")
    entry.bind("<Return>", lambda _event: seen.append(entry.get()))
    root.update()
    entry.focus_set()
    root.update()
    entry.event_generate("<Return>")
    root.update()
    root.destroy()
    return seen


print("visible          :", try_a_keypress(lambda r: None))
print("far off-screen   :", try_a_keypress(lambda r: r.geometry("200x80+5000+5000")))
print("withdrawn        :", try_a_keypress(lambda r: r.withdraw()))

Whether the keypress arrived depends on **which window has focus**, and focus is
assigned by the window manager — not by your program. Your own three answers may not
match the ones above, and that is the finding rather than a problem with it: a test
whose result depends on the compositor is a flaky test, and module 22 already said a
flaky test is worse than none.

So what a test client would have to fake is not the widget and not Tk. It is an
operating-system service: the pointer, the focus stack, the z-order. That is a larger
program than the one under test.

**The answer is architectural, and it is the part of this module worth keeping:**

> Put nothing in a callback that you cannot also call on its own.

`logic.py` holds the formatting and the limit decision. It imports no tkinter, so it
runs on a machine with no screen — which is why exercise 05 has no window in it at
all. `app.py` reads `logic.py` and does nothing else. And buttons are pressed with
`invoke()`, which calls the registered command and goes nowhere near the window
manager.

In [ ]:
from app import SensorWindow  # noqa: E402

root = tkinter.Tk()
root.withdraw()
window = SensorWindow(root)
window.location.set("Test rig")
window.refresh()
root.update()

print("rows        :", window.listing.size())
print("first row   :", window.listing.get(0))
print("status      :", window.status.get())

window.limit.set("90")
window.refresh()
print("at limit 90 :", window.status.get())

window.limit.set("warm")
window.refresh()
print("at 'warm'   :", window.status.get())
print("rows after  :", window.listing.size())
root.destroy()

That is the whole test strategy of the module, and the last line is deliberate: a
limit that is not a number left the listing as it was rather than emptying it. The
user typed something wrong; the answer to that is a message, not the loss of what was
on screen.

### The gap, stated honestly

`invoke()` is not blind — it honours the widget's own state. But it does not care
whether the widget is in the layout:

In [ ]:
root = tkinter.Tk()
root.withdraw()
fired: list[str] = []

disabled = tkinter.Button(
    root, text="d", command=lambda: fired.append("disabled"), state="disabled"
)
disabled.pack()

never_packed = tkinter.Button(root, text="n", command=lambda: fired.append("never packed"))
# no .pack() -- nobody can see this button, let alone click it

root.update()
disabled.invoke()
never_packed.invoke()

print("fired            :", fired)
print("is it on screen? :", bool(never_packed.winfo_ismapped()))
root.destroy()

Delete a `.pack()` line from `app.py` and **every test in `tests/` still passes**,
while the window has no button in it.

These tests check that the program computes the right thing and puts it in the right
widget. They cannot check that a person can see or reach that widget, and there is no
substitute for opening the window once and looking. Say so in a review rather than
claiming coverage you do not have.

## Where tkinter sits among the five

| | Flask (21) | Streamlit (22) | FastAPI (23) | tkinter (24) |
| --- | --- | --- | --- | --- |
| the caller | a browser | a browser | a program | a person at this machine |
| who owns the loop | the server | the server | the server | your process |
| control handed away | per request | per script run | per route call | once, for the program |
| bad input | your `if` | none to be bad | 422, before your code | your `try`, in the callback |
| a test client | `test_client()` | `AppTest` | `TestClient` | **none** |
| needs a network | yes | yes | yes | no |
| needs a display | no | no | no | **yes** |

**The heuristic: is the machine the user is at the machine the program runs on?** If
yes, a window needs no server, no port and no browser, and it keeps working when the
network is down — which on a shop floor is the requirement, not a nicety. If no, one
of the three above it.

---

`exercises/` is next: seven files to fill in and two to think through. Six of the
seven open a window; exercise 05 is the one that does not, and that is the point of
`logic.py`.

Module 25 does this in a terminal. There is no window manager to fight, the layout is
declarative, and it is the only one of the five that works down an ssh connection.